# Section 24.6 Diffusion models and DDIM sampling

Example 24-15, a diffusion model trained on scikit-learn's load_digits, and a self-contained copy of
Example 24-16, which retrains the same denoiser with the same seed and samples it with DDIM.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
torch.manual_seed(0)

X = torch.tensor(load_digits().images.reshape(-1, 64) / 16.0 * 2 - 1, dtype=torch.float32)  # [-1,1]

T = 50
t_lin = torch.linspace(0, 1, T + 1)
abar = torch.cos((t_lin + 0.008) / 1.008 * np.pi / 2) ** 2
abar = abar / abar[0]                                     # the cosine curve with value 1 at t = 0
print(f"unclipped last step: beta = {1 - abar[-1] / abar[-2]:.6f}, 1/sqrt(alpha) = {torch.sqrt(abar[-2] / abar[-1]):,.0f}")
beta = (1 - abar[1:] / abar[:-1]).clamp(max=0.999)        # every beta in (0, 1) (Nichol and Dhariwal, 2021)
alpha = 1 - beta
abar = torch.cumprod(alpha, 0)                            # abar for steps 1..T at indices 0..T-1
print(f"clipped last step:   beta = {beta[-1]:.6f}, 1/sqrt(alpha) = {1 / torch.sqrt(alpha[-1]):.1f}")
print(f"forward process signal fraction abar_t: step 1={abar[0]:.2f}, mid={abar[T//2]:.2f}, T={abar[-1]:.3f}")
print("by step T almost all signal is gone -> the data has become noise we know how to sample.")

class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(64 + 1, 256), nn.SiLU(),
                                 nn.Linear(256, 256), nn.SiLU(), nn.Linear(256, 256), nn.SiLU(),
                                 nn.Linear(256, 64))
    def forward(self, x, t):
        return self.net(torch.cat([x, t[:, None].float() / T], dim=1))

model = Denoiser(); opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for epoch in range(4000):
    idx = torch.randint(0, len(X), (256,)); x0 = X[idx]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    xt = torch.sqrt(abar[t])[:, None] * x0 + torch.sqrt(1 - abar[t])[:, None] * eps   # noise in one shot
    loss = nn.functional.mse_loss(model(xt, t), eps)        # learn to predict the noise
    opt.zero_grad(); loss.backward(); opt.step()
print(f"\ndenoiser trained: final noise-prediction MSE = {loss.item():.3f}")

@torch.no_grad()
def sample(n=2):
    x = torch.randn(n, 64)                                  # pure noise at step T
    for ti in reversed(range(T)):
        t = torch.full((n,), ti)
        eps = model(x, t)
        x0_hat = ((x - torch.sqrt(1 - abar[ti]) * eps) / torch.sqrt(abar[ti])).clamp(-1, 1)   # clean guess, in the data range
        abar_prev = abar[ti - 1] if ti > 0 else torch.tensor(1.0)
        mean = (torch.sqrt(abar_prev) * beta[ti] / (1 - abar[ti])) * x0_hat + (torch.sqrt(alpha[ti]) * (1 - abar_prev) / (1 - abar[ti])) * x
        var = (1 - abar_prev) / (1 - abar[ti]) * beta[ti]    # posterior variance, tilde beta_t
        x = mean + (torch.sqrt(var) * torch.randn_like(x) if ti > 0 else 0)
    return x

def show(vec):
    img = vec.reshape(8, 8)
    levels = " .:-=+*#"                                   # 8 intensity levels, dark -> bright
    for r in range(8):
        print("   " + "".join(levels[min(7, max(0, int((img[r, c] + 1) / 2 * 8)))] for c in range(8)))
print("\ntwo digits generated by denoising from pure noise:")
for s in sample(2).numpy():
    show(s); print()

print("All diffusion checks ran.")

unclipped last step: beta = 1.000000, 1/sqrt(alpha) = 712,891
clipped last step:   beta = 0.999000, 1/sqrt(alpha) = 31.6
forward process signal fraction abar_t: step 1=1.00, mid=0.46, T=0.000
by step T almost all signal is gone -> the data has become noise we know how to sample.



denoiser trained: final noise-prediction MSE = 0.181

two digits generated by denoising from pure noise:
       *=  
     .-**  
    :=*= : 
    =*-.== 
    -#+=#: 
     =:++. 
     . =+  
      .##  

     .++:  
     #+-*: 
    :#-:#. 
    :-.:*. 
    .:..*  
    .*:-#  
    .=-*=  
     .++.  

All diffusion checks ran.


In [2]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_digits
torch.manual_seed(0)

X = torch.tensor(load_digits().images.reshape(-1, 64) / 16.0 * 2 - 1, dtype=torch.float32)

T = 50
t_lin = torch.linspace(0, 1, T + 1)
abar = torch.cos((t_lin + 0.008) / 1.008 * np.pi / 2) ** 2
abar = abar / abar[0]                                     # the cosine curve with value 1 at t = 0
beta = (1 - abar[1:] / abar[:-1]).clamp(max=0.999)        # every beta in (0, 1) (Nichol and Dhariwal, 2021)
alpha = 1 - beta
abar = torch.cumprod(alpha, 0)                            # abar for steps 1..T at indices 0..T-1

class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(64 + 1, 256), nn.SiLU(),
                                 nn.Linear(256, 256), nn.SiLU(), nn.Linear(256, 256), nn.SiLU(),
                                 nn.Linear(256, 64))
    def forward(self, x, t):
        return self.net(torch.cat([x, t[:, None].float() / T], dim=1))

model = Denoiser(); opt = torch.optim.Adam(model.parameters(), lr=2e-3)
for epoch in range(4000):                                   # same training as Example 24-15
    idx = torch.randint(0, len(X), (256,)); x0 = X[idx]
    t = torch.randint(0, T, (256,))
    eps = torch.randn_like(x0)
    xt = torch.sqrt(abar[t])[:, None] * x0 + torch.sqrt(1 - abar[t])[:, None] * eps
    loss = nn.functional.mse_loss(model(xt, t), eps)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"denoiser trained: MSE = {loss.item():.3f}")

@torch.no_grad()
def ddim(x, steps=10):
    seq = torch.linspace(T - 1, 0, steps).round().long().tolist()   # short deterministic schedule
    for i, ti in enumerate(seq):
        eps = model(x, torch.full((x.shape[0],), ti))
        x0_pred = ((x - torch.sqrt(1 - abar[ti]) * eps) / torch.sqrt(abar[ti])).clamp(-1, 1)
        if i == len(seq) - 1:
            x = x0_pred                                     # last step: land on the clean estimate
        else:
            a_next = abar[seq[i + 1]]
            x = torch.sqrt(a_next) * x0_pred + torch.sqrt(1 - a_next) * eps   # NO added noise
    return x

def show(vec):
    img = vec.reshape(8, 8); levels = " .:-=+*#"
    for r in range(8):
        print("   " + "".join(levels[min(7, max(0, int((img[r, c] + 1) / 2 * 8)))] for c in range(8)))

torch.manual_seed(7); z = torch.randn(1, 64)
print("\nDDIM, 10 deterministic steps (Example 24-15 used 50 stochastic steps):")
show(ddim(z.clone(), steps=10)[0].numpy())
a = ddim(z.clone(), steps=10); b = ddim(z.clone(), steps=10)
print(f"\nsame z run twice, max abs difference = {(a - b).abs().max().item():.2e} (deterministic)")
print(f"network evaluations: DDIM 10 vs {T} for the stochastic sampler ({T // 10}x fewer).")

denoiser trained: MSE = 0.181

DDIM, 10 deterministic steps (Example 24-15 used 50 stochastic steps):
     :-+-  
     :==*. 
    .*::*: 
      .-*. 
    .=##:  
    :#:.:. 
     -. :. 
      .+-. 

same z run twice, max abs difference = 0.00e+00 (deterministic)
network evaluations: DDIM 10 vs 50 for the stochastic sampler (5x fewer).
